In [3]:
# !pip install torch 

In [ ]:
import torch
import sympy as sp

from dataset import EOS_ID, PAD_ID, SOS_ID, VOCAB_SIZE, encode, decode
from dataset_generation import _expr_to_prefix_tokens, prefix_tokens_to_infix
from model import CoeffPredTransformer


# ── Config ────────────────────────────────────────────────────────────────────
DEFAULT_CHECKPOINT = "models/v4_128_op_epoch_010.pt"
MAX_GEN_LEN        = 128


def load_model(checkpoint_path: str, device: torch.device) -> CoeffPredTransformer:
    """Load model weights and architecture from a checkpoint file."""
    ckpt = torch.load(checkpoint_path, map_location=device, weights_only=True)
    cfg  = ckpt["config"]

    model = CoeffPredTransformer(**cfg)
    model.load_state_dict(ckpt["model_state"])
    model.to(device)
    model.eval()

    print(f"  Loaded checkpoint : {checkpoint_path}")
    print(f"  Epoch             : {ckpt.get('epoch', '?')}")
    print(f"  Val loss          : {ckpt.get('val_loss', float('nan')):.6f}")
    print(f"  N coefficients    : {ckpt.get('n_coeffs', '?')}\n")
    return model

def predict(expression: str, model: CoeffPredTransformer, device: torch.device) -> str:
    """
    Tokenise a single expression string, run greedy decoding, and return
    the decoded prediction string.
    """
    # Parse expression string → sympy → prefix token list (matches training format)
    expr_sym = sp.sympify(expression)
    prefix_tokens = _expr_to_prefix_tokens(expr_sym)
    print(f"  Prefix tokens : {prefix_tokens}")

    # Encode with SOS/EOS wrapping, matching dataset.py src format:
    #   src = [SOS_ID] + encode(fn_tokens) + [EOS_ID]
    token_ids = [SOS_ID] + encode(prefix_tokens) + [EOS_ID]
    print(f"  Token IDs     : {token_ids}")
    src = torch.tensor([token_ids], dtype=torch.long).to(device)  # (1, L)

    with torch.no_grad():
        # generate_batch returns a list of token-id lists (one per batch item)
        pred_ids = model.generate_batch(src, max_len=MAX_GEN_LEN)[0]

    # Decode: skip_special=True removes PAD/SOS/EOS but keeps <BREAK>
    pred_tokens = decode(pred_ids, skip_special=True)
    pred_str = " ".join(pred_tokens)
    return pred_str


In [ ]:


checkpoint =  "models/v4_128_op_epoch_010.pt"
expr = "x*(sin(x) - 1/(3*x))"
# x*(-sin(x) - 1 + 1/x)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"  Device : {device}\n")

# ── Load model ────────────────────────────────────────────────────────
model = load_model(checkpoint, device)



In [8]:

# ── Run inference ─────────────────────────────────────────────────────
print(f"  Input expression : {expr}")
prediction = predict(expr, model, device)
print(f"  Predicted coeffs : {prediction}")
print()

# ── Pretty-print individual coefficients (split on <BREAK> token) ─────
# Denominators for Taylor series: n! for n = 0,1,2,3,4
FACTORIALS = [1, 1, 2, 6, 24]

coeff_strs = [c.strip() for c in prediction.split("<BREAK>")]
print(f"  Individual coefficients ({len(coeff_strs)} found):")

infix_coeffs = []
for i, c_str in enumerate(coeff_strs):
    c_tokens = c_str.split()
    c_infix  = prefix_tokens_to_infix(c_tokens)
    denom    = FACTORIALS[i] if i < len(FACTORIALS) else "?"
    print(f"    c[{i}]  prefix : {c_str}")
    print(f"    c[{i}]  infix  : {c_infix}   [n!= {denom}]")
    infix_coeffs.append(c_infix)



  Input expression : x*(sin(x) - 1/(3*x))
  Prefix tokens : ['*', 'x', '+', '*', '/', '-', '1', '+', '3', '**', 'x', '-', '1', 'sin', 'x']
  Token IDs     : [1, 18, 4, 6, 18, 19, 7, 9, 6, 11, 20, 4, 7, 9, 21, 4, 2]
  Predicted coeffs : + / - 1 + 3 * a sin a <BREAK> + * a cos a sin a <BREAK> + * + 2 cos a * * - 1 a sin a <BREAK> + * - 3 sin a * * - 1 a cos a <BREAK> + * - 4 cos a * a sin a

  Individual coefficients (5 found):
    c[0]  prefix : + / - 1 + 3 * a sin a
    c[0]  infix  : a*sin(a) - 1/3   [n!= 1]
    c[1]  prefix : + * a cos a sin a
    c[1]  infix  : a*cos(a) + sin(a)   [n!= 1]
    c[2]  prefix : + * + 2 cos a * * - 1 a sin a
    c[2]  infix  : -a*sin(a) + 2*cos(a)   [n!= 2]
    c[3]  prefix : + * - 3 sin a * * - 1 a cos a
    c[3]  infix  : -a*cos(a) - 3*sin(a)   [n!= 6]
    c[4]  prefix : + * - 4 cos a * a sin a
    c[4]  infix  : a*sin(a) - 4*cos(a)   [n!= 24]


In [9]:
# ── Complete Taylor series equation ────────────────────────────────────
print()
terms = []
for i, (c_infix, denom) in enumerate(zip(infix_coeffs, FACTORIALS)):
    coeff_part = f"({c_infix})/{denom}"
    if i == 0:
        terms.append(coeff_part)
    else:
        power_part = "(x-a)" if i == 1 else f"(x-a)^{i}"
        terms.append(f"{coeff_part}*{power_part}")

print(f"  Complete Taylor series:")
print(f"    f(x) ≈ {' + '.join(terms)}")



  Complete Taylor series:
    f(x) ≈ (a*sin(a) - 1/3)/1 + (a*cos(a) + sin(a))/1*(x-a) + (-a*sin(a) + 2*cos(a))/2*(x-a)^2 + (-a*cos(a) - 3*sin(a))/6*(x-a)^3 + (a*sin(a) - 4*cos(a))/24*(x-a)^4
